In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("SalesDataAnalysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 15:05:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.functions import col, when, count

In [3]:
df= spark.read.csv("/opt/spark-data/Sales/retail_sales_profiling.csv", header=True, inferSchema=True)

In [4]:
df.show(5)

+--------+----------+-----------+----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|order_id|order_date|customer_id|product_id|  product_name|   category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+--------+----------+-----------+----------+--------------+-----------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
| ORD1065|2024-12-21|   CUST0011|      P006|      Notebook| Stationery|          SP05|      Divya Nair| North|       3|     65.08|          15|          29.29|      165.95|    Debit Card|   Completed|
| ORD1196|2024-06-29|   CUST0019|      P004|Wireless Mouse|Electronics|          SP06|     Arjun Patel| South|       5|    929.24|          20|         929.24|     3716.96|    Debit Card|     Pend

In [5]:
null_counts = df.select([ count(when(col(c).isNull(), c)).alias(c) for c in df.columns ]) 
null_counts.show(truncate=False) 

+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|order_id|order_date|customer_id|product_id|product_name|category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|0       |0         |0          |0         |0           |0       |0             |25              |15    |0       |9         |42          |42             |9           |12            |0           |
+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+



In [6]:
from pyspark.sql.functions import mean, round as spark_round

In [7]:
df_nulls_fixed = df

In [8]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"discount_pct": 0.0, "discount_amount": 0.0})

In [9]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"salesperson_name": "Online / No Rep"})

In [10]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"region": "Unknown"})

In [11]:
df_nulls_fixed = df_nulls_fixed \
    .fillna({"payment_method": "Not Captured"})

In [13]:
null_counts_fixed = df_nulls_fixed.select([ count(when(col(c).isNull(), c)).alias(c) for c in df_nulls_fixed.columns ]) 
null_counts_fixed.show(truncate=False) 

+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|order_id|order_date|customer_id|product_id|product_name|category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|0       |0         |0          |0         |0           |0       |0             |0               |0     |0       |9         |0           |0              |9           |0             |0           |
+--------+----------+-----------+----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+



In [14]:
median_price_per_product = df_nulls_fixed \
    .filter(col("unit_price").isNotNull()) \
    .groupBy("product_id") \
    .agg(spark_round(mean(col("unit_price")), 2).alias("median_unit_price"))

In [15]:
median_price_per_product.show()

+----------+-----------------+
|product_id|median_unit_price|
+----------+-----------------+
|      P007|          8950.46|
|      P003|          6161.44|
|      P010|           138.19|
|      P006|            93.61|
|      P004|          1279.61|
|      P002|          25909.1|
|      P001|          66770.5|
|      P008|           514.37|
|      P009|          2783.67|
|      P005|         11065.27|
+----------+-----------------+



In [16]:
df_nulls_fixed = df_nulls_fixed \
    .join(median_price_per_product, on="product_id", how="left") \
    .withColumn(
        "unit_price",
        when(col("unit_price").isNull(), col("median_unit_price"))
        .otherwise(col("unit_price"))
    ) \
    .drop("median_unit_price")

In [17]:
df_nulls_fixed = df_nulls_fixed \
    .withColumn(
        "total_amount",
        when(
            col("total_amount").isNull(),
            spark_round(
                col("unit_price") * col("quantity") - col("discount_amount"), 2
            )
        ).otherwise(col("total_amount"))
    )

In [18]:
null_counts_fixed = df_nulls_fixed.select([ count(when(col(c).isNull(), c)).alias(c) for c in df_nulls_fixed.columns ]) 
null_counts_fixed.show(truncate=False) 

+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|product_id|order_id|order_date|customer_id|product_name|category|salesperson_id|salesperson_name|region|quantity|unit_price|discount_pct|discount_amount|total_amount|payment_method|order_status|
+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+
|0         |0       |0         |0          |0           |0       |0             |0               |0     |0       |0         |0           |0              |0           |0             |0           |
+----------+--------+----------+-----------+------------+--------+--------------+----------------+------+--------+----------+------------+---------------+------------+--------------+------------+

